# Session 1: Modern LLM Foundations for Agentic Systems (Student Notebook)

Welcome to Session 1! In this notebook, you will explore the foundational concepts behind Large Language Models (LLMs) and learn how to interact with them programmatically. These skills form the bedrock for building agentic AI systems in subsequent sessions.

## Learning Objectives

By the end of this session, you will be able to:

1. **Set up and configure** LLM API connections using the OpenAI Python SDK
2. **Understand tokenization** and how context windows affect LLM behavior
3. **Control model output** by tuning parameters such as temperature, top_p, and max_tokens
4. **Use system messages** to shape LLM response behavior and persona
5. **Build basic LLM pipelines** by chaining multiple API calls together
6. **Lay the groundwork** for conversational agents with message history management

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import tiktoken
import os
import json

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or uncomment the line below and paste your key (not recommended for production):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: Setting Up LLM API Connections

In this demo we initialize the OpenAI client and make our first chat completion call. The `client` object handles authentication, request formatting, and response parsing for us.

In [ ]:
# Demo 1 - Setting Up LLM API Connections

# Step 1: Initialize the OpenAI client
# The client automatically reads OPENAI_API_KEY from the environment.
client = openai.OpenAI()

# Step 2: Make a simple chat completion call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is an LLM? Explain in two sentences."}
    ],
    max_tokens=100
)

# Step 3: Print the response
print("Model used :", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\nAssistant response:")
print(response.choices[0].message.content)

### Demo 2: Understanding Tokenization and Context Windows

Tokens are the fundamental units that LLMs process. A token can be a word, part of a word, or even a single character. Understanding tokenization helps you:
- Estimate API costs (billing is per-token)
- Stay within context window limits
- Write more efficient prompts

In [ ]:
# Demo 2 - Understanding Tokenization and Context Windows

# Step 1: Get the tokenizer for gpt-4o-mini
encoder = tiktoken.encoding_for_model("gpt-4o-mini")

# Step 2: Encode various texts and count tokens
texts = [
    "Hello, world!",
    "Artificial Intelligence is transforming the way we build software.",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilisticexpialidocious",
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)"
]

print("Token counts for different texts:")
print("=" * 60)
for text in texts:
    tokens = encoder.encode(text)
    print(f"\nText   : {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Chars  : {len(text)}")
    print(f"Tokens : {len(tokens)}")
    print(f"Token IDs (first 10): {tokens[:10]}")

# Step 3: Demonstrate context window limits
print("\n" + "=" * 60)
print("Context window reference:")
print("  gpt-4o-mini : 128,000 tokens")
print("  gpt-4o      : 128,000 tokens")
print("  gpt-3.5-turbo: 16,385 tokens")

# Show how a long text would consume the window
long_text = "This is a sample sentence. " * 500
long_tokens = encoder.encode(long_text)
print(f"\nLong text ({len(long_text)} chars) = {len(long_tokens)} tokens")
print(f"That is {len(long_tokens)/128000*100:.2f}% of the gpt-4o-mini context window.")

### Demo 3: Exploring Model Parameters

LLMs expose several parameters that control the randomness and length of generated text:

| Parameter | Description |
|-----------|-------------|
| `temperature` | Controls randomness. 0 = deterministic, 1 = creative |
| `top_p` | Nucleus sampling. Lower values = more focused output |
| `max_tokens` | Maximum number of tokens in the response |

In [ ]:
# Demo 3 - Exploring Model Parameters

client = openai.OpenAI()
prompt = "Write a one-sentence description of the future of AI."

# --- Temperature comparison ---
print("TEMPERATURE COMPARISON")
print("=" * 60)

for temp in [0.0, 0.5, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=80
    )
    print(f"\nTemperature {temp}:")
    print(f"  {response.choices[0].message.content}")

# --- top_p comparison ---
print("\n" + "=" * 60)
print("TOP_P COMPARISON")
print("=" * 60)

for top_p in [0.1, 0.5, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        top_p=top_p,
        max_tokens=80
    )
    print(f"\ntop_p {top_p}:")
    print(f"  {response.choices[0].message.content}")

# --- max_tokens comparison ---
print("\n" + "=" * 60)
print("MAX_TOKENS COMPARISON")
print("=" * 60)

for max_tok in [10, 30, 100]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=max_tok
    )
    text = response.choices[0].message.content
    print(f"\nmax_tokens={max_tok} (finish_reason={response.choices[0].finish_reason}):")
    print(f"  {text}")

### Demo 4: Comparing LLM Response Behaviors

The **system message** is a powerful lever for controlling how the LLM behaves. By changing the system message, you can make the same model act as different "personas" or agents, each with its own tone, expertise, and communication style.

In [ ]:
# Demo 4 - Comparing LLM Response Behaviors

client = openai.OpenAI()

user_question = "Explain what a REST API is."

personas = {
    "Formal Academic": (
        "You are a formal academic professor. Use precise technical language, "
        "cite concepts from computer science, and maintain a scholarly tone."
    ),
    "Casual Friendly": (
        "You are a casual, friendly mentor. Use simple language, analogies, "
        "and a conversational tone. Imagine you are explaining to a friend."
    ),
    "Technical DevOps Engineer": (
        "You are a senior DevOps engineer. Focus on practical implementation details, "
        "mention specific tools and protocols, and be concise."
    ),
}

print(f"User Question: {user_question}")
print("=" * 60)

for persona_name, system_msg in personas.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_question}
        ],
        temperature=0.7,
        max_tokens=200
    )
    print(f"\n--- {persona_name} ---")
    print(response.choices[0].message.content)
    print()

### Demo 5: Building a Basic LLM Pipeline

A pipeline chains multiple LLM calls together, where the output of one step becomes the input of the next. This pattern is fundamental to agentic systems, where an LLM may need to plan, execute, and reflect across multiple steps.

In [ ]:
# Demo 5 - Building a Basic LLM Pipeline

client = openai.OpenAI()

def call_llm(system_message, user_message, temperature=0.5, max_tokens=300):
    """Helper function to make an LLM call with a system and user message."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


# --- Pipeline: Summarize -> Extract Key Points ---

original_text = """
Large Language Models (LLMs) have rapidly evolved from simple text generators to sophisticated 
reasoning engines. Modern LLMs like GPT-4, Claude, and Gemini can understand context, follow 
complex instructions, and even use tools. This evolution has enabled the emergence of agentic AI 
systems -- autonomous programs that can plan, reason, and take actions to accomplish goals. These 
agents leverage LLMs as their core reasoning component while integrating with external tools, 
databases, and APIs. The key challenges in building agentic systems include managing context 
windows, ensuring reliable tool use, implementing effective guardrails, and maintaining 
conversational state across multi-turn interactions. As the field progresses, we are seeing 
the rise of multi-agent architectures where multiple specialized agents collaborate to solve 
complex problems that no single agent could handle alone.
"""

print("PIPELINE STEP 1: Summarize")
print("=" * 60)

summary = call_llm(
    system_message="You are a concise summarizer. Summarize the given text in 2-3 sentences.",
    user_message=f"Summarize this text:\n\n{original_text}"
)
print(summary)

print("\nPIPELINE STEP 2: Extract Key Points")
print("=" * 60)

key_points = call_llm(
    system_message="You are a structured information extractor. Extract exactly 5 key points as a numbered list.",
    user_message=f"Extract the key points from this summary:\n\n{summary}"
)
print(key_points)

print("\nPIPELINE RESULT: Complete output")
print("=" * 60)
print(f"Original length : {len(original_text.split())} words")
print(f"Summary length  : {len(summary.split())} words")
print(f"\nKey Points:\n{key_points}")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders. Fill in the code to complete each task.

### Task 1: Configure and Test Multiple LLM Provider Connections

Create a function that initializes an OpenAI client, sends a simple test message, and returns the response. This verifies that your API key and network connection are working correctly.

In [ ]:
# Task 1 - Configure and Test Multiple LLM Provider Connections

# TODO: Create a function that initializes an OpenAI client and tests the connection
# by sending a simple "Hello" message and returning the response.
#
# Hint: Use openai.OpenAI() to create the client
# Hint: Use client.chat.completions.create() with model="gpt-4o-mini"
# Hint: Return response.choices[0].message.content

def test_llm_connection():
    """Initialize OpenAI client and test with a simple message."""
    # YOUR CODE HERE
    pass


# Test your function
# result = test_llm_connection()
# print(result)

### Task 2: Analyze Token Usage and Optimize Prompt Length

Write a function that counts the tokens in a prompt and, if the token count exceeds a given limit, truncates the prompt intelligently at sentence boundaries so it fits within the budget.

In [ ]:
# Task 2 - Analyze Token Usage and Optimize Prompt Length

# TODO: Write a function that takes a prompt string, counts its tokens using tiktoken,
# and if it exceeds a given limit, truncates it intelligently (at sentence boundaries).
#
# Hint: Use tiktoken.encoding_for_model("gpt-4o-mini") to get the encoder
# Hint: Use encoder.encode(text) to get tokens
# Hint: Split by sentences (e.g., split on ". ") and rebuild until under the limit

def optimize_prompt(prompt, max_tokens=500):
    """Count tokens and truncate at sentence boundaries if over the limit."""
    # YOUR CODE HERE
    pass


# Test your function
# long_prompt = "This is sentence one. " * 200
# optimized = optimize_prompt(long_prompt, max_tokens=50)
# print(optimized)

### Task 3: Experiment with Model Parameters

Create a function that sends the same prompt at three different temperature settings and returns all three responses so you can compare how randomness affects output.

In [ ]:
# Task 3 - Experiment with Model Parameters

# TODO: Create a function that sends the same prompt with 3 different temperature
# settings (0.0, 0.5, 1.0) and returns all three responses for comparison.
#
# Hint: Use a loop over temperature values [0.0, 0.5, 1.0]
# Hint: Store results in a dictionary {temperature: response_text}
# Hint: Use the same prompt and model for fair comparison

def compare_temperatures(prompt):
    """Send the same prompt at different temperatures and return all responses."""
    # YOUR CODE HERE
    pass


# Test your function
# results = compare_temperatures("Describe the color blue in a creative way.")
# for temp, text in results.items():
#     print(f"Temperature {temp}: {text}\n")

### Task 4: Build a Simple Conversational Agent Foundation

Build a `SimpleAgent` class that maintains a conversation history, uses a system message to define its persona, and can hold multi-turn conversations with a user.

In [ ]:
# Task 4 - Build a Simple Conversational Agent Foundation

# TODO: Build a simple conversational agent that:
# 1. Maintains a message history list
# 2. Has a system message defining it as a helpful coding assistant
# 3. Takes user input, appends to history, sends to LLM, appends response
# 4. Returns the assistant's response
#
# Hint: Start with messages = [{"role": "system", "content": "..."}]
# Hint: Append user messages with {"role": "user", "content": user_input}
# Hint: After getting response, append {"role": "assistant", "content": response}

class SimpleAgent:
    def __init__(self):
        """Initialize the agent with a system message and empty history."""
        # YOUR CODE HERE
        pass

    def chat(self, user_input):
        """Process a user message and return the assistant's response."""
        # YOUR CODE HERE
        pass

    def get_history(self):
        """Return the full conversation history."""
        # YOUR CODE HERE
        pass


# Test your agent
# agent = SimpleAgent()
# print(agent.chat("What is a Python decorator?"))
# print("---")
# print(agent.chat("Can you show me a simple example?"))
# print("---")
# print(f"History length: {len(agent.get_history())} messages")

---
## Summary

In this session you learned the foundational skills for working with LLMs programmatically:

1. **API Connections** -- How to initialize the OpenAI client and make chat completion requests.
2. **Tokenization** -- How text is converted to tokens, why token counts matter for cost and context window limits, and how to use `tiktoken` to measure them.
3. **Model Parameters** -- How `temperature`, `top_p`, and `max_tokens` influence the style, creativity, and length of LLM outputs.
4. **System Messages & Personas** -- How the system message acts as a powerful control lever to change LLM behavior without changing the user prompt.
5. **LLM Pipelines** -- How to chain multiple LLM calls together so the output of one step feeds into the next, forming the basis for agentic workflows.

**Up Next:** In Session 2, we will dive into advanced prompting techniques -- including few-shot prompting, chain-of-thought reasoning, and structured output generation -- that make LLMs more reliable and useful as the reasoning core of agentic systems.